In [110]:
import os 
import random 
import torch 
from d2l import torch as d2l

In [111]:
d2l.DATA_HUB['wikitext-2'] = ('https://s3.amazonaws.com/research.metamind.io/wikitext/wikitext-2-v1.zip',
                            '3c914d17d80b1459be871a5039ac23e752a53cbe')

def _read_wiki(data_dir):
    file_name = os.path.join(data_dir, 'wiki.train.tokens')
    with open(file_name, 'r', encoding='UTF-8') as f:   # 每行是一个段落
        lines = f.readlines()
    # 大写字母转换为小写字母
    # 句子按照句号分隔
    # 只保留有至少两个句子的段落
    paragraphs = [line.strip().lower().split(' . ') for line in lines if len(line.split(' . ')) >= 2]
    random.shuffle(paragraphs)
    return paragraphs

为预训练任务定义辅助函数

生成下一句预测任务的数据

In [112]:
def _get_next_sentence(sentence, next_sentence, paragraphs):
    """
    50%概率生成原语句+下一句, 50%概率生成原语句+随机语句
    
    Parameters
    -----------
        sentence : 当前语句
        next_sentence : 当前语句的下一句
        paragraphs : 全部段落
    
    Returns
    --------
        sentence : 当前语句
        next_sentence :  50%概率当前语句的下一句, 50%概率随机语句
        is_next : True, 表示当前语句的下一句; False, 表示随机语句
    """
    if random.random() < 0.5:
        is_next = True
    else:
        # paragraphs是三重列表的嵌套
        next_sentence = random.choice(random.choice(paragraphs))     # 从全部段落中随机抽出一句话
        is_next = False
    return sentence, next_sentence, is_next

In [113]:
def _get_nsp_data_from_paragraph(paragraph, paragraphs, vocab, max_len):
    """
    从一个paragraph段落生成用于下一句预测的训练样本

    Parameters
    -----------
        paragraph : 一个段落
        paragraphs : 全部段落
        vocab : 词表
        max_len : 指定了输入序列，即段落的词元最大数量(过长的被过滤)
    
    Return
    -------
        nsp_data_from_paragraph : 该paragraph段落生成的所有下一句预测任务的样本(tokens, segments, is_next)
    """
    nsp_data_from_paragraph = []
    for i in range(len(paragraph) - 1):
        tokens_a, tokens_b, is_next = _get_next_sentence(paragraph[i], paragraph[i+1], paragraphs)
        # 3 考虑1个'<cls>'词元和2个'<sep>'词元
        if len(tokens_a) + len(tokens_b) + 3 > max_len:   # max_len指定了输入序列，即段落的词元最大数量
            continue
        # tokens 文本序列， segments[0,1]列表表示文本序列a,b的标记
        tokens, segments = d2l.get_tokens_and_segments(tokens_a, tokens_b)
        nsp_data_from_paragraph.append((tokens, segments, is_next))
    return nsp_data_from_paragraph

生成掩蔽语言模型任务的数据

In [114]:
# mlm masked language modeling
def _replace_mlm_tokens(tokens, candidate_pred_positions, num_mlm_preds, vocab):
    """
    为掩蔽语言模型的输入创建新的词元副本, 其中输入可能包含替换的'<mask>'词元或随机词元

    Parameters
    -----------
        tokens : 词元的列表
        candidate_pred_positions : # 候选预测词元编号(即标记所有非特殊词元, 这些属于待预测词元)
        num_mlm_preds : 预测的数量
        vocab : 词表
    
    Returns
    --------
        mlm_input_tokens : 词元列表, 需要掩蔽的词元已掩蔽
        pred_positions_and_labels : 列表, 每个元素是(待预测词元编号, 待预测词元[即掩蔽前的词元存档])
    """
    mlm_input_tokens = [token for token in tokens]   # 词元列表
    pred_positions_and_labels = []   # (候选预测词元编号, 候选预测词元)
    # 打乱后用于在掩蔽语言模型任务中获取15%的随机词元进行预测
    random.shuffle(candidate_pred_positions)
    for mlm_pred_position in candidate_pred_positions:   # 对每个候选预测词元
        if len(pred_positions_and_labels) >= num_mlm_preds:   # 满足预测数量后，停止
            break 
        masked_token = None                                   # 初始化      
        # 80%的情况，将词元替换为'<mask>'词元，即掩蔽
        if random.random() < 0.8:
            masked_token = '<mask>'
        else:
            # 10%的情况，保持原词元不变
            if random.random() < 0.5:
                masked_token = tokens[mlm_pred_position]
            # 10%的情况，用随机词元替换该词
            else:
                masked_token = random.choice(vocab.idx_to_token)
        mlm_input_tokens[mlm_pred_position] = masked_token # 替换成掩蔽
        pred_positions_and_labels.append((mlm_pred_position, tokens[mlm_pred_position]))
    return mlm_input_tokens, pred_positions_and_labels

In [115]:
def _get_mlm_data_from_tokens(tokens, vocab):
    """
    获取生成掩蔽模型任务的数据
    
    Parameters
    -----------
        tokens : BERT输入序列文本, 元素是词元
        vocab : 词表
    
    Returns
    --------
        vocab[mlm_input_tokens] : 词元编号(已部分被掩蔽)
        pred_positions : 发生预测的词元编号(即做出掩蔽的编号)
        vocab[mlm_pred_labels] : 预测的标签编号(即做出掩蔽的真实值)
    """
    candidate_pred_positions = []     # 候选预测词元，用编号表示
    # tokens 是一个字符串列表
    for i, token in enumerate(tokens):
        # 在掩蔽语言模型任务中不会预测特殊词元
        if token in ['<cls>', '<sep>']:
            continue 
        candidate_pred_positions.append(i)
    # 掩蔽语言模型任务中预测15%的随机词元
    num_mlm_preds = max(1, round(len(tokens) * 0.15))
    mlm_input_tokens, pred_positions_and_labels = _replace_mlm_tokens(tokens, candidate_pred_positions, num_mlm_preds, vocab)
    pred_positions_and_labels = sorted(pred_positions_and_labels, key=lambda x: x[0])   # 根据编号升序排序词元列表
    pred_positions = [v[0] for v in pred_positions_and_labels]    # 待预测词元编号                   
    mlm_pred_labels = [v[1] for v in pred_positions_and_labels]   #  待预测词元[即掩蔽前的词元存档]
    return vocab[mlm_input_tokens], pred_positions, vocab[mlm_pred_labels]

将文本转换为预训练数据集

In [116]:
def _pad_bert_inputs(examples, max_len, vocab):
    """
    将不足max_len长度的样本数据进行填充
    
    Parameters 
    -----------
        examples : 下一句预测及掩蔽这两个任务的样本数据
        max_len : 指定了输入序列，即段落的词元最大数量(过长的被过滤)
    
    Returns
    --------
        all_token_ids : 词元编号(已部分被掩蔽), 不足max_len长度的用'<pad>'填充
        all_segments : [0,1]列表表示文本序列a,b的标记, 不足max_len长度的用0填充
        valid_lens : 词元编号(已部分被掩蔽), 不包含填充词的有效长度
        all_pred_positions : 发生预测的词元编号(即做出掩蔽的编号), 不足max_len * 0.15长度的用0填充
        all_mlm_weights : [0, 1]列表表示pred_positions中填充与否的标记, 0填充, 1非填充
        all_mlm_labels : 预测的标签编号(即做出掩蔽的真实值), 不足max_len * 0.15长度的用0填充
        nsp_labels : True, 表示当前语句的下一句; False, 表示随机语句
    """
    max_num_mlm_preds = round(max_len * 0.15)     # 最大一个段落中掩蔽词元数
    all_token_ids, all_segments, valid_lens, = [], [], []
    all_pred_positions, all_mlm_weights, all_mlm_labels = [], [], []
    nsp_labels = []
    # examples[i]中共5个元素
    # token_ids : 词元编号(已部分被掩蔽)
    # pred_positions : 发生预测的词元编号(即做出掩蔽的编号)
    # mlm_pred_label_ids : 预测的标签编号(即做出掩蔽的真实值)
    # segments : [0,1]列表表示文本序列a,b的标记
    # is_next : True, 表示当前语句的下一句; False, 表示随机语句
    for (token_ids, pred_positions, mlm_pred_label_ids, segments, is_next) in examples:
        all_token_ids.append(torch.tensor(token_ids + [vocab['<pad>']] * (max_len-len(token_ids)), dtype=torch.long))
        all_segments.append(torch.tensor(segments + [0] * (max_len - len(segments)), dtype=torch.long))
        # valid_lens不包括'<pad>'的计数
        valid_lens.append(torch.tensor(len(token_ids), dtype=torch.float32))
        all_pred_positions.append(torch.tensor(pred_positions + [0] * (max_num_mlm_preds - len(pred_positions)), dtype=torch.long))
        # 填充词元的预测将通过乘以权重0在损失中过滤掉
        all_mlm_weights.append(torch.tensor([1.0] * len(mlm_pred_label_ids) + [0.0] * (max_num_mlm_preds - len(pred_positions)), dtype=torch.float32))
        all_mlm_labels.append(torch.tensor(mlm_pred_label_ids + [0] * (max_num_mlm_preds - len(mlm_pred_label_ids)), dtype=torch.long))
        nsp_labels.append(torch.tensor(is_next, dtype=torch.long))
    return (all_token_ids, all_segments, valid_lens, all_pred_positions, all_mlm_weights, all_mlm_labels, nsp_labels)

In [117]:
class _WikiTextDataset(torch.utils.data.Dataset):
    def __init__(self, paragraphs, max_len):
        # 输入paragraphs[i]是代表段落的句子字符串列表
        # 而输出paragraphs[i]是代表段落的句子列表，其中每个句子都是词元列表
        paragraphs = [d2l.tokenize(paragraph, token='word') for paragraph in paragraphs] # 将句子字符串进一步拆分为词元列表
        sentences = [sentence for paragraph in paragraphs for sentence in paragraph]     # 每个元素是句子(已拆分为词元列表)
        self.vocab = d2l.Vocab(sentences, min_freq=5, reserved_tokens=['<pad>', '<mask>', '<cls>', '<sep>'])  # 词表
        # 获取下一句预测任务的数据
        examples = []   
        for paragraph in paragraphs:   # 生成的所有下一句预测任务的样本(tokens, segments, is_next)
            examples.extend(_get_nsp_data_from_paragraph(paragraph, paragraphs, self.vocab, max_len))
        # 获取掩蔽语言模型任务的数据
        # tokens, 前句+后句的文本序列，segments前后句的标记，is_next是否为真实后句的标记
        examples = [(_get_mlm_data_from_tokens(tokens, self.vocab) + (segments, is_next)) for tokens, segments, is_next in examples]
        # examples[i]中共5个元素
        # vocab[mlm_input_tokens] : 词元编号(已部分被掩蔽)
        # pred_positions : 发生预测的词元编号(即做出掩蔽的编号)
        # vocab[mlm_pred_labels] : 预测的标签编号(即做出掩蔽的真实值)
        # segments : [0,1]列表表示文本序列a,b的标记
        # is_next : True, 表示当前语句的下一句; False, 表示随机语句
        
        # 填充输入
        (self.all_token_ids, self.all_segments, self.valid_lens, self.all_pred_positions,
         self.all_mlm_weights, self.all_mlm_labels, self.nsp_labels) = _pad_bert_inputs(examples, max_len, self.vocab)
    
    def __getitem__(self, idx):
        return (self.all_token_ids[idx], self.all_segments[idx], self.valid_lens[idx], self.all_pred_positions[idx],
                self.all_mlm_weights[idx], self.all_mlm_labels[idx], self.nsp_labels[idx])

    def __len__(self):
        return len(self.all_token_ids)

In [118]:
def load_data_wiki(batch_size, max_len):
    """
    加载WikiText-2数据集
    
    Parameters
    -----------
        batch_size : 批量大小
        max_len : 指定了输入序列，即段落的词元最大数量(过长的被过滤)
    
    Returns
    --------
        train_iter : 训练数据迭代器
        train_set.vocab : 词表
    """
    data_dir = d2l.download_extract('wikitext-2', 'wikitext-2')
    paragraphs = _read_wiki(data_dir)        # 加载数据集中所有段落，段落中按句子拆分(一个列表是一个段落，列表中元素是一个句子字符串)
    train_set = _WikiTextDataset(paragraphs, max_len)       # 加载训练数据，包括下一句预测及掩蔽词元任务
    print(train_set)
    train_iter = torch.utils.data.DataLoader(train_set, batch_size, shuffle=True)   # 训练数据迭代器
    return train_iter, train_set.vocab

In [119]:
batch_size, max_len = 512, 64
train_iter, vocab = load_data_wiki(batch_size, max_len)
for (tokens_X, segments_X, valid_lens_x, pred_positions_X, mlm_weights_X, mlm_Y, nsp_y) in train_iter:
    print(tokens_X.shape, segments_X.shape, valid_lens_x.shape, pred_positions_X.shape, mlm_weights_X.shape, mlm_Y.shape, nsp_y.shape)
    break

..\data\wikitext-2-v1.zip
torch.Size([512, 64]) torch.Size([512, 64]) torch.Size([512]) torch.Size([512, 10]) torch.Size([512, 10]) torch.Size([512, 10]) torch.Size([512])


In [120]:
len(vocab)

20256

NLTK(natural language toolkit)句子拆分

In [122]:
import nltk 
nltk.download('punkt')

[nltk_data] Error loading punkt: <urlopen error [Errno 11004]
[nltk_data]     getaddrinfo failed>


False

In [127]:
sentences = 'This is great ! Why not ?'
nltk.tokenize.sent_tokenize(sentences)

['This is great !', 'Why not ?']